In [1]:
import os
import shutil
import re
import pandas as pd
import json

#### Move batch files to data folder

In [3]:
data_folder = 'data'
os.makedirs(data_folder, exist_ok=True)

for root, dirs, files in os.walk('.'):
    for dir_name in dirs:
        if 'batch' in dir_name:
            batch_folder_path = os.path.join(root, dir_name)
            for file_name in os.listdir(batch_folder_path):
                if file_name.endswith('.txt') or file_name.endswith('.ann'):
                    file_path = os.path.join(batch_folder_path, file_name)
                    shutil.move(file_path, data_folder)
                    print(f"Moved {file_path} to {data_folder}")

#### Move .ann Data

In [4]:
data_folder = 'corpora/data'
os.makedirs(data_folder, exist_ok=True)
lct_txt = 'corpora/lct_ann'

for file_name in os.listdir(data_folder):
    if file_name.endswith('.ann'):
        file_path = os.path.join(data_folder, file_name)
        shutil.copy(file_path, lct_txt)
        print(f"Moved {file_path} to {lct_txt}")

## Splitte Inc und Exc aus allen Daten

In [1]:
def read_criteria_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

def write_to_file(file_path, content):
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(content)


def process_files(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    unsplit_files = []

    for filename in os.listdir(input_folder):
        if filename.endswith(".txt"):
            nct_number = re.findall(r'NCT\d+', filename)
            if not nct_number:
                nct_number = re.findall(r'nct\d+', filename, re.IGNORECASE)
                if not nct_number:
                    unsplit_files.append(filename)
                    continue
            nct_number = nct_number[0]

            file_path = os.path.join(input_folder, filename)
            content = read_criteria_file(file_path)

            inclusion_criteria = ""
            exclusion_criteria = ""
            current_section = None

            # Split sections using regex to identify inclusion and exclusion sections
            sections = re.split(r'(Inclusion Criteria:|Exclusion Criteria:)', content, flags=re.IGNORECASE)
            for i in range(1, len(sections), 2):
                section_header = sections[i].strip().lower()
                section_content = sections[i + 1].strip()

                if "inclusion criteria" in section_header:
                    inclusion_criteria += section_content + "\n"
                elif "exclusion criteria" in section_header:
                    exclusion_criteria += section_content + "\n"

            inclusion_criteria = "\n".join([line.lstrip() for line in inclusion_criteria.split('\n')])
            exclusion_criteria = "\n".join([line.lstrip() for line in exclusion_criteria.split('\n')])

            # Write inclusion criteria to file, create an empty file if no inclusion criteria
            inc_filename = f"{nct_number}_inc.txt"
            inc_file_path = os.path.join(output_folder, inc_filename)
            write_to_file(inc_file_path, inclusion_criteria.strip())

            
            # Write exclusion criteria to file, create an empty file if no exclusion criteria
            exc_filename = f"{nct_number}_exc.txt"
            exc_file_path = os.path.join(output_folder, exc_filename)
            write_to_file(exc_file_path, exclusion_criteria.strip())

    # Print out the filenames that couldn't be split into inc and exc
    if unsplit_files:
        print("Files that couldn't be split into inclusion and exclusion criteria:")
        for file in unsplit_files:
            print(file)



def count_files_in_folders(input_folder, output_folder):
    input_count = len([name for name in os.listdir(input_folder) if os.path.isfile(os.path.join(input_folder, name))])
    output_count = len([name for name in os.listdir(output_folder) if os.path.isfile(os.path.join(output_folder, name))])

    print(f"Number of files in {input_folder}: {input_count}")
    print(f"Number of files in {output_folder}: {output_count}")

In [2]:
input_folder = "corpora/lct_txt" #"corpora/lct_txt"  # "corpora/lct_p1"
output_folder = "corpora/lct_txt_half"   #"corpora/lct_p1_half"
process_files(input_folder, output_folder)
# 1987 Dateien mit Inhalt und insgesammt 2012 Datein

In [48]:
count_files_in_folders(input_folder, output_folder)

### Welche Files Fehlen in lct_p1_half

In [42]:
def get_nct_numbers(folder):
    nct_numbers = set()
    for filename in os.listdir(folder):
        match = re.search(r'NCT\d+', filename)
        if match:
            nct_numbers.add(match.group())
    return nct_numbers

input_folder = "corpora/lct_p1_geprüft"
output_folder = "corpora/lct_p1_half"

input_nct_numbers = get_nct_numbers(input_folder)
output_nct_numbers = get_nct_numbers(output_folder)
print(input_nct_numbers)
print("Number of NCT numbers in input folder:", len(input_nct_numbers))
print(output_nct_numbers)
missing_nct_numbers = input_nct_numbers - output_nct_numbers
print("Number of missing NCT numbers in output folder:", len(output_nct_numbers))
print("Missing NCT numbers in output folder:")
for nct_number in sorted(missing_nct_numbers):
    print(nct_number)

In [26]:
# 986 Dateien
# ec und ic -> 
# NCT03860181

## Count Operators in JSON Files

In [1]:
def count_operators(directory):
    and_count = 0
    or_count = 0
    not_count = 0

    json_files = [f for f in os.listdir(directory) if f.endswith('.json')]

    for file in json_files:
        file_path = os.path.join(directory, file)
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            for entry in data:
                if entry['type'] == 'And':
                    and_count += 1
                elif entry['type'] == 'Or':
                    or_count += 1
                elif entry['type'] == 'Negation':
                    not_count += 1

    return and_count, or_count, not_count

directory = 'corpora/all_operators'
and_count, or_count, not_count = count_operators(directory)

print(f"Total NOT operators: {not_count}")
print(f"Total AND operators: {and_count}")
print(f"Total OR operators: {or_count}")

### Count AND OR NOT in ann files

In [1]:
def count_operators(directory):
    negation_with_negates_count = 0
    negation_with_two_args_count = 0
    negation_with_offsets_count = 0
    and_matches_list = []
    or_matches_list = []
    negation_with_negates_list = []
    negation_with_two_args_list = []
    negation_with_offsets_list = []
    and_count = 0
    or_count = 0

    ann_files = [f for f in os.listdir(directory) if f.endswith('.ann')]

    for file in ann_files:
        file_path = os.path.join(directory, file)
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
            # Find Negations with offsets and a word
            negation_with_offsets_matches = re.findall(r'T\d+\s+Negation\s+\d+\s+\d+\s+\w+', content)
            negation_with_offsets_list.extend(negation_with_offsets_matches)
            negation_with_offsets_count += len(negation_with_offsets_matches)

            # Find specific AND operators like "R6 And Arg1:E7 Arg2:E2" or "R6 And Arg1:T7 Arg2:T2"
            and_matches = re.findall(r'R\d+\s+And\s+Arg1:(?:E|T)\d+\s+Arg2:(?:E|T)\d+\b', content)
            and_matches_list.extend(and_matches)
            and_count += len(and_matches)
            #print(and_matches)

            # Find specific OR operators like "R7 Or Arg1:E10 Arg2:E12" or "R7 Or Arg1:T10 Arg2:T12"
            or_matches = re.findall(r'R\d+\s+Or\s+Arg1:(?:E|T)\d+\s+Arg2:(?:E|T)\d+\b', content)
            or_matches_list.extend(or_matches)
            or_count += len(or_matches)

    return (negation_with_negates_count, negation_with_two_args_count,
            negation_with_offsets_count, and_count, or_count,
            and_matches_list, or_matches_list,
            negation_with_negates_list, negation_with_two_args_list,
            negation_with_offsets_list)

directory = 'corpora/lct_ann'
(negation_with_negates_count, negation_with_two_args_count,
 negation_with_offsets_count, and_count, or_count,
 and_matches_list, or_matches_list,
 negation_with_negates_list, negation_with_two_args_list,
 negation_with_offsets_list) = count_operators(directory)


print(f"Total NOT : {negation_with_offsets_count}")
print(f"Total AND operators: {and_count}")
print(f"Total OR operators: {or_count}")

print("\nAND matches:")
#for match in and_matches_list:
#    print(match)

# print("\nOR matches:")
# for match in or_matches_list:
#     print(match)
# 
# print("\nNegation matches with offsets and a word:")
# for match in negation_with_offsets_list:
#     print(match)

## Find Operators in P1 Files

In [3]:
def count_operators_in_files(directory):
    operators_count = {'[AND]': 0, '[OR]': 0, '[NOT]': 0}
    # Durchlaufe alle Dateien im angegebenen Verzeichnis
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):  # Angenommen, die Dateien sind .txt-Dateien
            file_path = os.path.join(directory, filename)
            with open(file_path, 'r', encoding="utf-8") as file:
                content = file.read()
                # Zähle jeden Operator in der Datei
                for operator in operators_count.keys():
                    operators_count[operator] += content.count(operator)

    return operators_count

directory = 'corpora/lct_p1_geprüft'
operator_counts = count_operators_in_files(directory)
print(operator_counts)

### Find missing annotations (alt)

In [8]:
def count_operators(directory):
    and_matches_dict = {}
    or_matches_dict = {}

    ann_files = [f for f in os.listdir(directory) if f.endswith('.ann')]

    for file in ann_files:
        file_path = os.path.join(directory, file)
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
            and_matches = re.findall(r'R\d+\s+And\s+Arg1:(?:E|T)\d+\s+Arg2:(?:E|T)\d+\b', content)
            or_matches = re.findall(r'R\d+\s+Or\s+Arg1:(?:E|T)\d+\s+Arg2:(?:E|T)\d+\b', content)

            if and_matches:
                and_matches_dict[file] = and_matches
            if or_matches:
                or_matches_dict[file] = or_matches

    return and_matches_dict, or_matches_dict

def read_ann_files(directory):
    ann_files = [f for f in os.listdir(directory) if f.endswith('.ann')]
    data = {}
    for file in ann_files:
        file_path = os.path.join(directory, file)
        with open(file_path, 'r', encoding='utf-8') as f:
            data[file] = f.read()
    return data

def parse_ann_file(content):
    lines = content.strip().split('\n')
    df = pd.DataFrame([line.split('\t', 1) for line in lines], columns=['ID', 'Details'])

    rel_df = df[df['ID'].str.startswith('R')]
    and_pattern = re.compile(r'^(R\d+)\tAnd Arg1:(E\d+|T\d+) Arg2:(E\d+|T\d+)$')
    or_pattern = re.compile(r'^(R\d+)\tOr Arg1:(E\d+|T\d+) Arg2:(E\d+|T\d+)$')
    relationships = []
    for index, row in rel_df.iterrows():
        line = f"{row['ID']}\t{row['Details']}"
        and_match = and_pattern.match(line)
        if and_match:
            rel_id, arg1, arg2 = and_match.groups()
            relationships.append(('And', arg1, arg2))
        else:
            or_match = or_pattern.match(line)
            if or_match:
                rel_id, arg1, arg2 = or_match.groups()
                relationships.append(('Or', arg1, arg2))

    return relationships

directory = 'corpora/lct_ann'
and_matches_dict, or_matches_dict = count_operators(directory)
ann_files = read_ann_files(directory)

missing_and_files = []
missing_or_files = []

for file, content in ann_files.items():
    relationships = parse_ann_file(content)
    and_relationships = [rel for rel in relationships if rel[0] == 'And']
    or_relationships = [rel for rel in relationships if rel[0] == 'Or']

    if file in and_matches_dict and not and_relationships:
        missing_and_files.append(file)
    if file in or_matches_dict and not or_relationships:
        missing_or_files.append(file)

print("Files with missing AND operators:")
for file in missing_and_files:
    print(file)

print("\nFiles with missing OR operators:")
for file in missing_or_files:
    print(file)


#### 5 largest p3 files

In [15]:
def count_entities(data):
    def count_entities_recursive(node):
        if isinstance(node, dict):
            conditions = len(node.get("Condition", []))
            drugs = len(node.get("Drug", []))
            observations = len(node.get("Observation", []))

            for value in node.values():
                if isinstance(value, dict):
                    child_conditions, child_drugs, child_observations = count_entities_recursive(value)
                    conditions += child_conditions
                    drugs += child_drugs
                    observations += child_observations

            return conditions, drugs, observations

        return 0, 0, 0

    conditions, drugs, observations = count_entities_recursive(data)
    return conditions + drugs + observations

def process_json_files(folder_path):
    file_counts = {}

    for file_name in os.listdir(folder_path):
        if file_name.endswith(".json"):
            file_path = os.path.join(folder_path, file_name)

            with open(file_path, 'r', encoding="utf-8") as file:
                data = json.load(file)

            entity_count = count_entities(data)
            file_counts[file_name] = entity_count

    return file_counts

def copy_top_files(folder_path, file_counts, top_n):
    sorted_files = sorted(file_counts.items(), key=lambda x: x[1], reverse=True)

    for i in range(min(top_n, len(sorted_files))):
        file_name = sorted_files[i][0]
        json_source_path = os.path.join(folder_path, file_name)
        json_destination_path = os.path.join("top_files", file_name)
        shutil.copy(json_source_path, json_destination_path)

        txt_file_name = file_name.split("_p3")[0] + ".txt"
        print(txt_file_name)
        txt_source_path = os.path.join("corpora", "lct_txt_half", txt_file_name)
        txt_destination_path = os.path.join("top_files", txt_file_name)
        shutil.copy(txt_source_path, txt_destination_path)

folder_path = 'corpora/lct_p3_half'
file_counts = process_json_files(folder_path)

top_n = 5
os.makedirs("n_shot_files", exist_ok=True)
copy_top_files(folder_path, file_counts, top_n)

print(f"The top {top_n} files with the most Condition, Drug, and Observation entities have been copied to the 'top_files' folder.")
print(f"The corresponding text files from 'corpora/lct_txt_half' have also been copied to the 'top_files' folder.")